# Figures 5D, 5E, 5F & Figure S13: Hippocampal-ACC Theta Synchrony (PPC & PSI) and Reinstatement Correlations

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig 5D (exemplar raw + theta-filtered 4-8 Hz HPC and ACC traces), Fig 5E (HPC-ACC theta PPC by RPE tercile at cue; 500 cut-and-reverse surrogates; mixed-effects regression p_pos:neg = 0.002, p_pos:neutral = 0.09), Fig 5F (HPC->ACC theta PSI by RPE tercile at cue; Morlet 4-8 Hz, 0.5 Hz steps, 4 cycles, 500 cut-and-reverse surrogates; mixed-effects regression p_pos:neg = 0.04, p_pos:neutral = 0.2), Fig S13 (subject-mean positive-RPE cue PPC vs ACC RPE reactivation r = -0.33 and vs ACC-HPC hit RSA r = 0.43; all ps > 0.05) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/projects/guLab/Salman/EphysAnalyses/{subj}/neural/Day1/cue_on-epo.fif`, `Day1_reref_elec_df`, `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv`, `ACC_rpe_category_HFA_subject_reinstatement.csv`, `group_results_ACC_to_HPC.pkl` |
| **Outputs** | `ppc_pairs_cue_on.csv`, `all_psi_pairs_cue_on_HPC_to_ACC_from_trial_cache.csv`, `figures/Fig5DEF_S13_PPC_PSI/{ppc_bar_strip_cue_on_HPC_to_ACC.pdf,psi_bar_strip_cue_on_HPC_to_ACC.pdf,ppc_cue_on_vs_ACC_reinstatement.pdf,ppc_cue_on_vs_ACC_HPC_hitRSA.pdf}` |
| **Dependencies** | `membandit`, `mne`, `mne-connectivity`, `statsmodels`, R `lme4` (`R/lmer_neural.R`) |
| **Expected runtime** | ~1-2 hours from raw epochs (500 surrogates/pair); seconds when reading per-subject `.npz` caches |
| **Upstream steps** | `04_decoding_rsa_followups/Fig4_S8_RSA.ipynb` and `matlab/test_script_crossdecoding.m` (for Section 6 / Fig S13); run `Rscript R/lmer_neural.R` for R `lmer` fits |


# HPC-ACC theta synchrony by RPE category (PPC headline)

Parsimonious pipeline:

- Pairwise HPC-ACC theta PPC (4-8 Hz) by RPE tercile (`negative` / `neutral` / `positive`)
- Surrogate normalization with cut-and-reverse evoked surrogate null (make_surrogate_evoked_data)
- Subject-level omnibus = mean PPC-z across channel pairs
- Pair-level mixed-effects model: `ppc_z ~ rpe_cat` (reference = `positive`)

Follow-up analyses added:

- Manuscript-quality example raw + theta-filtered traces for high-PPC HPC→ACC trials
- Multi-target (`ACC`, `AMY`, `OFC`, `dlPFC`, `dmPFC`) bar+strip PPC summary (HPC as source)
- Trial-wise HPC→ACC theta PSI (`phase_slope_index_time`) for cue-trial hit/miss × RPE (Sections 9–10)


## 0. Setup


In [ ]:
# membandit setup
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import wilcoxon

import mne
import mne_connectivity as _mne_conn
from mne import EpochsArray
from mne_connectivity import spectral_connectivity_epochs, seed_target_indices
import statsmodels.formula.api as smf

from membandit.plotting_style import set_plot_style
from membandit.behavior import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
)
from membandit.io import load_subject_epochs
from membandit.granger import build_rpe_trial_map
from membandit.phase_sync import zscore_vs_evoked_surrogate, compute_epoch_psi, zscore_epoch_psi
from membandit.subject_covariates import (
    load_acc_reinstatement_table,
    load_hpc_memory_classification_table,
    load_acc_hpc_hit_rsa_table,
    correlate_subject_metric,
)

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")
set_plot_style()

print("mne-connectivity", _mne_conn.__version__)


## 1. Config


In [ ]:
EVENT = "cue_on"  # "feedback_start" or "cue_on"

EVENT_SPECS = {
    "feedback_start": dict(tmin=0.0, tmax=1.5, behavior="learn_df"),
    "cue_on": dict(tmin=0.0, tmax=1.0, behavior="combined_df"),
}
if EVENT not in EVENT_SPECS:
    raise ValueError(f"EVENT must be one of {list(EVENT_SPECS)}")

SPEC = EVENT_SPECS[EVENT]
TMIN = float(SPEC["tmin"])
TMAX = float(SPEC["tmax"])
BEHAVIOR_SOURCE = SPEC["behavior"]

DAY = "Day1"
ROI_SOURCE = "HPC"
ROI_TARGET = "ACC"
FMIN, FMAX = 4.0, 8.0
TARGET_SFREQ = 50.0
MIN_TRIALS = 5
N_SURROGATES = 500
N_JOBS = 4
SEED = 42
RPE_CATS = ("negative", "neutral", "positive")
COLORS = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}
OVERWRITE_CACHE = False

base_dir = paths.get_base_dir()
load_dir = f"{base_dir}/work/qasims01/MemoryBanditData/EMU"
savepath = f"{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/ppc_hpc_acc_rpe_cutrev/{EVENT}"
plot_savepath = os.path.join(paths.figure_output_root(), "Fig5DEF_S13_PPC_PSI")
try:
    os.makedirs(plot_savepath, exist_ok=True)
except OSError:
    from pathlib import Path as _Path
    plot_savepath = os.path.join(str(_Path.cwd()), "figures", "MemoryBandit-PPC_HPC_ACC_RPE")
    os.makedirs(plot_savepath, exist_ok=True)
os.makedirs(savepath, exist_ok=True)

PAD = 1.0
CWT_FREQS = np.arange(4.0, 8.5, 0.5)
CWT_N_CYCLES = 4.0
SMOKE = False  # True: process one subject only (dev)

savepath_psi = f"{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi_hpc_acc_rpe_hitmiss/{EVENT}"
os.makedirs(savepath_psi, exist_ok=True)


print(f"EVENT={EVENT} window={TMIN}–{TMAX}s behavior={BEHAVIOR_SOURCE}")
print(f"cache: {savepath}")
print(f"N_SURROGATES={N_SURROGATES} N_JOBS={N_JOBS}")
print(f"PSI trial cache: {savepath_psi}")


## 2. Subjects and behavior


In [ ]:
demo_file = f"{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx"
demo_data = pd.read_excel(demo_file, engine="openpyxl")
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df["MSSMCode"].isin(["MS015", "MS024", "MS034", "MS038"])]
subjects = sorted(subj_df["MSSMCode"].dropna().unique().tolist())

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)
beh_df = learn_df if BEHAVIOR_SOURCE == "learn_df" else combined_df

print("Subjects:", len(subjects))
print("behavior source:", BEHAVIOR_SOURCE, beh_df.shape)
print("rpe_cat values:", sorted(beh_df["rpe_cat"].dropna().astype(str).unique().tolist()))


## 3. PPC helpers + per-subject cache


In [ ]:
def _unique_keep_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _pool_roi_channels(epoch_data):
    # load_subject_epochs keys follow GC_HEMI_KEYS: "all" (current) or "l"/"r" (legacy).
    def _collect(prefix):
        chans = list(epoch_data.get(f"{prefix}_channels_all") or [])
        if not chans:
            chans = list(epoch_data.get(f"{prefix}_channels_l") or []) + list(
                epoch_data.get(f"{prefix}_channels_r") or []
            )
        return _unique_keep_order(chans)

    hpc = _collect("source")
    acc = _collect("target")
    shared = set(hpc) & set(acc)
    if shared:
        acc = [ch for ch in acc if ch not in shared]
    return hpc, acc


def _extract_pair_values(conn):
    vals = np.asarray(conn.get_data(), dtype=float)
    vals = np.squeeze(vals)
    if vals.ndim == 0:
        vals = vals.reshape(1)
    if vals.ndim > 1:
        vals = np.nanmean(vals, axis=-1)
    return np.asarray(vals, dtype=float)


def _compute_ppc(arr, info, indices, sfreq):
    ep = EpochsArray(arr, info.copy(), tmin=float(info["sfreq"] * 0.0), verbose=False)
    conn = spectral_connectivity_epochs(
        ep,
        method="ppc",
        indices=indices,
        mode="multitaper",
        sfreq=sfreq,
        fmin=FMIN,
        fmax=FMAX,
        faverage=True,
        n_jobs=1,
        verbose=False,
    )
    return _extract_pair_values(conn)


def cache_path(subj_id):
    return os.path.join(savepath, f"{subj_id}_HPC_ACC_ppc_evsurr.npz")


def process_subject(subj_id):
    path = cache_path(subj_id)
    if os.path.exists(path) and not OVERWRITE_CACHE:
        z = np.load(path, allow_pickle=True)
        return {k: z[k] for k in z.files}

    epoch_data = load_subject_epochs(
        subj_id=subj_id,
        base_dir=base_dir,
        event=EVENT,
        roi_source=ROI_SOURCE,
        roi_target=ROI_TARGET,
        tmin=None,
        tmax=None,
        target_sfreq=TARGET_SFREQ,
        day=DAY,
        fmin=None,
        fmax=None,
    )
    if epoch_data is None:
        return None

    hpc, acc = _pool_roi_channels(epoch_data)
    if len(hpc) == 0 or len(acc) == 0:
        print(f"  skip {subj_id}: missing HPC ({len(hpc)}) or ACC ({len(acc)})")
        return None

    epochs = epoch_data["epochs"].copy().crop(tmin=TMIN, tmax=TMAX)
    epochs_roi = epochs.copy().pick(hpc + acc, verbose=False)
    hpc_idx = np.array([epochs_roi.ch_names.index(ch) for ch in hpc], dtype=int)
    acc_idx = np.array([epochs_roi.ch_names.index(ch) for ch in acc], dtype=int)
    indices = seed_target_indices(hpc_idx, acc_idx)
    pair_src = np.array([epochs_roi.ch_names[i] for i in indices[0]], dtype=str)
    pair_tgt = np.array([epochs_roi.ch_names[i] for i in indices[1]], dtype=str)
    pair_label = np.array([f"{s}->{t}" for s, t in zip(pair_src, pair_tgt)], dtype=str)
    sfreq = float(epochs_roi.info["sfreq"])
    info = epochs_roi.info

    trial_map = build_rpe_trial_map(
        beh_df, subj_id, n_epochs=len(epochs_roi), day=DAY, event=EVENT
    )
    trial_map = trial_map[trial_map["alignment_valid"]].copy()

    out = {
        "subj_id": np.array(subj_id),
        "pair_src": pair_src,
        "pair_tgt": pair_tgt,
        "pair_label": pair_label,
        "n_hpc": np.array(len(hpc)),
        "n_acc": np.array(len(acc)),
        "n_pairs": np.array(len(pair_label)),
    }

    n_ok = 0
    for cat in RPE_CATS:
        idx = (
            trial_map.loc[trial_map["rpe_cat"] == cat, "epoch_idx"]
            .astype(int)
            .drop_duplicates()
            .to_numpy()
        )
        out[f"n_trials_{cat}"] = np.array(len(idx))
        if len(idx) < MIN_TRIALS:
            continue

        arr = epochs_roi[idx].get_data(copy=True)
        ppc_true = _compute_ppc(arr=arr, info=info, indices=indices, sfreq=sfreq)

        def _conn_fn(x):
            return _compute_ppc(arr=x, info=info, indices=indices, sfreq=sfreq)

        ppc_z, surr_mean, surr_std = zscore_vs_evoked_surrogate(
            conn_fn=_conn_fn,
            data=arr,
            info=info,
            n_surrogates=N_SURROGATES,
            n_jobs=N_JOBS,
            seed=SEED,
        )
        out[f"ppc_{cat}"] = np.asarray(ppc_true, dtype=float)
        out[f"ppc_z_{cat}"] = np.asarray(ppc_z, dtype=float)
        out[f"surr_mean_{cat}"] = np.asarray(surr_mean, dtype=float)
        out[f"surr_std_{cat}"] = np.asarray(surr_std, dtype=float)
        n_ok += 1

    if n_ok == 0:
        print(f"  skip {subj_id}: no RPE category reached MIN_TRIALS={MIN_TRIALS}")
        return None

    np.savez_compressed(path, **out)
    return out


## 4. Run per-subject caching + group summary plot


In [ ]:
results = {}
t0 = time.time()
for subj_id in tqdm(subjects, desc="PPC HPC→ACC"):
    rec = process_subject(subj_id)
    if rec is not None:
        results[subj_id] = rec
print(f"Finished {len(results)} subjects in {(time.time() - t0) / 60:.1f} min")

keep = [s for s, rec in results.items() if all(f"ppc_z_{cat}" in rec for cat in RPE_CATS)]
print(f"Complete-case N = {len(keep)} / {len(results)}")
if len(keep) < 2:
    raise RuntimeError("Need at least 2 complete-case subjects")

subj_omni = {
    cat: np.array(
        [
            np.nanmean(np.asarray(results[s][f"ppc_z_{cat}"], dtype=float))
            for s in keep
        ],
        dtype=float,
    )
    for cat in RPE_CATS
}

group_path = os.path.join(savepath, f"group_{EVENT}_HPC_ACC_ppc.npz")
np.savez_compressed(
    group_path,
    subjects=np.array(keep, dtype=str),
    **{f"ppc_omni_z_{cat}": subj_omni[cat] for cat in RPE_CATS},
)
print("saved", group_path)

print("\nOne-sample Wilcoxon vs 0")
for cat in RPE_CATS:
    stat = wilcoxon(subj_omni[cat], zero_method="wilcox", alternative="two-sided")
    print(f"{cat}: W={stat.statistic:.3f}, p={stat.pvalue:.5f}")

pair_keys = [("negative", "positive"), ("negative", "neutral"), ("positive", "neutral")]
print("\nPaired Wilcoxon contrasts")
for a, b in pair_keys:
    stat = wilcoxon(subj_omni[a], subj_omni[b], zero_method="wilcox", alternative="two-sided")
    print(f"{a} - {b}: W={stat.statistic:.3f}, p={stat.pvalue:.5f}")

x = np.arange(len(RPE_CATS), dtype=float)
fig, ax = plt.subplots(figsize=(4.6, 4.4))
mat = np.column_stack([subj_omni[c] for c in RPE_CATS])
for row in mat:
    ax.plot(x, row, color="0.78", lw=0.9, alpha=0.9)
means = np.nanmean(mat, axis=0)
sems = np.nanstd(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0])
for i, cat in enumerate(RPE_CATS):
    ax.errorbar(
        x[i], means[i], yerr=sems[i], fmt="o", ms=8, capsize=3,
        color=COLORS[cat], label=cat, zorder=5,
    )
ax.axhline(0.0, color="k", ls="--", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(RPE_CATS))
ax.set_ylabel("PPC z (mean over HPC-ACC pairs)")
ax.tick_params(axis="both", which="major", length=4, width=1.0, bottom=True, left=True)
ax.set_title(f"HPC-ACC theta PPC · {EVENT} · N={len(keep)}")
ax.legend(frameon=False, loc="best")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig_path = os.path.join(plot_savepath, f"ppc_hpc_acc_{EVENT}_omnibus.pdf")
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print("saved figure", fig_path)
plt.show()



## 5. Pair-level MixedLM


In [ ]:
rows = []
for subj_id in keep:
    rec = results[subj_id]
    pair_label = np.asarray(rec["pair_label"]).astype(str)
    pair_src = np.asarray(rec["pair_src"]).astype(str)
    pair_tgt = np.asarray(rec["pair_tgt"]).astype(str)
    for cat in RPE_CATS:
        vals = np.asarray(rec[f"ppc_z_{cat}"], dtype=float)
        for pi, v in enumerate(vals):
            if not np.isfinite(v):
                continue
            rows.append(
                {
                    "subject": str(subj_id),
                    "pair_label": str(pair_label[pi]),
                    "src_ch": str(pair_src[pi]),
                    "tgt_ch": str(pair_tgt[pi]),
                    "rpe_cat": str(cat),
                    "ppc_z": float(v),
                }
            )

df_mm = pd.DataFrame(rows)
df_mm["rpe_cat"] = df_mm["rpe_cat"].astype(str)
df_mm["pair_uid"] = df_mm["subject"].astype(str) + ":" + df_mm["pair_label"].astype(str)

print(
    "rows:",
    len(df_mm),
    "subjects:",
    df_mm["subject"].nunique(),
    "pair_labels:",
    df_mm["pair_label"].nunique(),
    "pair_uid:",
    df_mm["pair_uid"].nunique(),
)

csv_path = os.path.join(savepath, f"ppc_pairs_{EVENT}.csv")
df_mm.to_csv(csv_path, index=False)
print("saved", csv_path)

r_stub = f'''
# R / lme4 starter code
library(lme4)

df <- read.csv("{csv_path}")
df$rpe_cat <- relevel(factor(df$rpe_cat), ref = "positive")

# Preferred model: random slope of rpe_cat across subjects + pair random intercept
m <- lmer(ppc_z ~ rpe_cat + (1 + rpe_cat | subject) + (1 | pair_uid), data = df)

# If singular/non-convergent, fallback:
# m <- lmer(ppc_z ~ rpe_cat + (1 | subject) + (1 | pair_uid), data = df)

summary(m)
'''
print(r_stub)


In [ ]:
import re

# Variant 1: hemisphere-matched pairs only (left-left or right-right).
def infer_hemi(ch_name):
    s = str(ch_name).strip().lower()
    if re.search(r"(^|[^a-z0-9])(left|lh|l)([^a-z0-9]|$)", s):
        return "L"
    if re.search(r"(^|[^a-z0-9])(right|rh|r)([^a-z0-9]|$)", s):
        return "R"
    return np.nan


df_ppc_hemi = df_mm.copy()
df_ppc_hemi["src_hemi"] = df_ppc_hemi["src_ch"].map(infer_hemi)
df_ppc_hemi["tgt_hemi"] = df_ppc_hemi["tgt_ch"].map(infer_hemi)

same_hemi = (
    df_ppc_hemi["src_hemi"].isin(["L", "R"])
    & df_ppc_hemi["tgt_hemi"].isin(["L", "R"])
    & (df_ppc_hemi["src_hemi"] == df_ppc_hemi["tgt_hemi"])
)
df_ppc_hemi = df_ppc_hemi.loc[same_hemi].copy()

hemi_csv_path = os.path.join(savepath, f"ppc_pairs_{EVENT}_hemi_matched.csv")
df_ppc_hemi.to_csv(hemi_csv_path, index=False)
print(
    "hemi-matched rows:",
    len(df_ppc_hemi),
    "subjects:",
    df_ppc_hemi["subject"].nunique(),
    "pairs:",
    df_ppc_hemi["pair_uid"].nunique(),
)
print("saved", hemi_csv_path)

# Variant 2: use continuous zrpe predictor (subject x rpe_cat mean zrpe).
required_cols = {"participant", "rpe_cat", "zrpe"}
missing_cols = required_cols - set(beh_df.columns)
if missing_cols:
    raise ValueError(f"beh_df is missing required columns for zrpe variant: {sorted(missing_cols)}")

beh_for_zrpe = beh_df.copy()
if "condition" in beh_for_zrpe.columns and "DAY" in globals():
    beh_for_zrpe = beh_for_zrpe.loc[beh_for_zrpe["condition"].astype(str) == str(DAY)].copy()

beh_for_zrpe["rpe_cat"] = beh_for_zrpe["rpe_cat"].astype(str).str.lower()
beh_for_zrpe["participant"] = beh_for_zrpe["participant"].astype(str)
beh_for_zrpe["zrpe"] = pd.to_numeric(beh_for_zrpe["zrpe"], errors="coerce")

zrpe_map = (
    beh_for_zrpe.dropna(subset=["participant", "rpe_cat", "zrpe"])
    .groupby(["participant", "rpe_cat"], as_index=False)["zrpe"]
    .mean()
    .rename(columns={"participant": "subject", "zrpe": "zrpe_mean_cat"})
)

# Keep one PPC row per subject/pair/rpe_cat and add matched continuous zrpe.
df_ppc_zrpe = df_mm.copy()
df_ppc_zrpe["rpe_cat"] = df_ppc_zrpe["rpe_cat"].astype(str).str.lower()
df_ppc_zrpe = df_ppc_zrpe.merge(zrpe_map, on=["subject", "rpe_cat"], how="left")
df_ppc_zrpe = df_ppc_zrpe.dropna(subset=["zrpe_mean_cat"]).copy()

zrpe_csv_path = os.path.join(savepath, f"ppc_pairs_{EVENT}_zrpe.csv")
df_ppc_zrpe.to_csv(zrpe_csv_path, index=False)
print(
    "zrpe rows:",
    len(df_ppc_zrpe),
    "subjects:",
    df_ppc_zrpe["subject"].nunique(),
    "pairs:",
    df_ppc_zrpe["pair_uid"].nunique(),
)
print("saved", zrpe_csv_path)

print(
    """
R model stubs
-------------
# Hemisphere-matched categorical model
# m_hemi <- lmer(ppc_z ~ rpe_cat + (1 + rpe_cat | subject) + (1 | pair_uid), data = df_ppc_hemi)

# Continuous zrpe model
# m_zrpe <- lmer(ppc_z ~ zrpe_mean_cat + (1 + zrpe_mean_cat | subject) + (1 | pair_uid), data = df_ppc_zrpe)
"""
)


## 6. Subject-level PPC vs ACC reinstatement and ACC-HPC RSA

Mean subject-level PPC-z per RPE category (`ppc_z_mean`) is correlated against:

- ACC RPE reinstatement (`ACC_MeanAccuracy`)
- ACC-HPC hit RSA (`ACC_HPC_HitRSA_cluster_mean`)
- HPC memory classification (`HPC_MemoryClassAccuracy`)


In [ ]:
ppc_subject = pd.DataFrame(
    [
        {"Subject": s, "rpe_cat": cat, "ppc_z_mean": subj_omni[cat][i]}
        for i, s in enumerate(keep)
        for cat in RPE_CATS
    ]
)
print(
    "subject x rpe rows:", len(ppc_subject),
    "subjects:", ppc_subject["Subject"].nunique(),
)

reinstatement_tbl = load_acc_reinstatement_table(base_dir=base_dir)
rsa_subject_tbl = load_acc_hpc_hit_rsa_table(base_dir=base_dir, n_jobs=N_JOBS)
hpc_mem_tbl = load_hpc_memory_classification_table(base_dir=base_dir)
print("ACC reinstatement N =", len(reinstatement_tbl))
print("ACC-HPC hit RSA N =", len(rsa_subject_tbl))
print("HPC memory classification N =", len(hpc_mem_tbl))

ppc_acc_merged, ppc_acc_corr = correlate_subject_metric(
    x_tbl=ppc_subject,
    metric_tbl=reinstatement_tbl,
    x_col="ppc_z_mean",
    x_label=f"{EVENT} HPC→ACC theta PPC (z)",
    y_col="ACC_MeanAccuracy",
    y_label="ACC RPE reactivation (%)",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"ppc_{EVENT}_vs_ACC_reinstatement",
    title=f"ACC RPE reactivation vs {EVENT} HPC→ACC theta PPC",
)
ppc_rsa_merged, ppc_rsa_corr = correlate_subject_metric(
    x_tbl=ppc_subject,
    metric_tbl=rsa_subject_tbl,
    x_col="ppc_z_mean",
    x_label=f"{EVENT} HPC→ACC theta PPC (z)",
    y_col="ACC_HPC_HitRSA_cluster_mean",
    y_label="ACC–HPC hit RSA (cluster mean)",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"ppc_{EVENT}_vs_ACC_HPC_hitRSA",
    title=f"ACC–HPC hit RSA vs {EVENT} HPC→ACC theta PPC",
)
ppc_hpcmem_merged, ppc_hpcmem_corr = correlate_subject_metric(
    x_tbl=ppc_subject,
    metric_tbl=hpc_mem_tbl,
    x_col="ppc_z_mean",
    x_label=f"{EVENT} HPC→ACC theta PPC (z)",
    y_col="HPC_MemoryClassAccuracy",
    y_label="HPC memory classification (hit/miss accuracy)",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"ppc_{EVENT}_vs_HPC_memory_classification",
    title=f"HPC memory classification vs {EVENT} HPC→ACC theta PPC",
)


## 7. Manuscript-quality PPC examples (raw + theta-filtered traces)

Rank subjects by peak positive-RPE pair PPC-z, take the top 4, and plot 10 high phase-locked positive-RPE cue trials (distributed 3/3/2/2 across subjects). Each column shows raw traces with 4–8 Hz overlays for that subject's best HPC→ACC pair.


In [ ]:
from scipy.signal import hilbert

N_EXAMPLE_SUBJECTS = 4
N_EXAMPLES = 10


def _rank_subject_pairs(results_dict, subjects_keep):
    rows = []
    for subj_id in subjects_keep:
        rec = results_dict.get(subj_id)
        if rec is None or "ppc_z_positive" not in rec:
            continue
        vals = np.asarray(rec["ppc_z_positive"], dtype=float)
        if vals.size == 0 or np.all(~np.isfinite(vals)):
            continue
        idx = int(np.nanargmax(vals))
        rows.append(
            {
                "subject": str(subj_id),
                "pair_idx": idx,
                "score": float(vals[idx]),
                "src": str(np.asarray(rec["pair_src"])[idx]),
                "tgt": str(np.asarray(rec["pair_tgt"])[idx]),
            }
        )
    rows.sort(key=lambda r: r["score"], reverse=True)
    return rows


def _single_trial_phase_locking(hpc_filt, acc_filt):
    ph_hpc = np.angle(hilbert(hpc_filt))
    ph_acc = np.angle(hilbert(acc_filt))
    dphi = ph_hpc - ph_acc
    return float(np.abs(np.nanmean(np.exp(1j * dphi))))


def _top_trials_for_pair(pair_info, n_take):
    epoch_data = load_subject_epochs(
        subj_id=pair_info["subject"],
        base_dir=base_dir,
        event=EVENT,
        roi_source=ROI_SOURCE,
        roi_target=ROI_TARGET,
        tmin=None,
        tmax=None,
        target_sfreq=TARGET_SFREQ,
        day=DAY,
        fmin=None,
        fmax=None,
    )
    if epoch_data is None:
        return []

    epochs = epoch_data["epochs"].copy().crop(tmin=TMIN, tmax=TMAX)
    pair_chs = [pair_info["src"], pair_info["tgt"]]
    for ch in pair_chs:
        if ch not in epochs.ch_names:
            raise RuntimeError(f"Example channel not found in epochs: {ch}")

    epochs_pair = epochs.copy().pick(pair_chs, verbose=False)
    trial_map = build_rpe_trial_map(
        beh_df,
        pair_info["subject"],
        n_epochs=len(epochs_pair),
        day=DAY,
        event=EVENT,
    )
    pos_idx = (
        trial_map.loc[
            trial_map["alignment_valid"] & trial_map["rpe_cat"].eq("positive"),
            "epoch_idx",
        ]
        .astype(int)
        .drop_duplicates()
        .to_numpy()
    )
    if len(pos_idx) == 0:
        return []

    raw = epochs_pair[pos_idx].get_data(copy=True)
    sfreq = float(epochs_pair.info["sfreq"])
    filt = mne.filter.filter_data(
        raw,
        sfreq=sfreq,
        l_freq=FMIN,
        h_freq=FMAX,
        method="fir",
        verbose=False,
    )
    times = epochs_pair.times.copy()

    trial_scores = []
    for i in range(raw.shape[0]):
        sc = _single_trial_phase_locking(filt[i, 0], filt[i, 1])
        trial_scores.append((i, sc))
    trial_scores.sort(key=lambda x: x[1], reverse=True)

    out = []
    for i, sc in trial_scores[: int(n_take)]:
        out.append(
            {
                "subject": pair_info["subject"],
                "src": pair_info["src"],
                "tgt": pair_info["tgt"],
                "ppc_z_pair": pair_info["score"],
                "epoch_idx": int(pos_idx[i]),
                "phase_lock": float(sc),
                "times": times,
                "hpc_raw": raw[i, 0],
                "acc_raw": raw[i, 1],
                "hpc_f": filt[i, 0],
                "acc_f": filt[i, 1],
            }
        )
    return out


ranked_pairs = _rank_subject_pairs(results, keep)
if len(ranked_pairs) == 0:
    raise RuntimeError("Could not find any valid positive-RPE PPC pair")

top_pairs = ranked_pairs[: min(N_EXAMPLE_SUBJECTS, len(ranked_pairs))]
print(f"Top {len(top_pairs)} subjects by positive-RPE pair PPC-z:")
for r in top_pairs:
    print(
        f"  {r['subject']}: pair PPC-z={r['score']:.3f}  "
        f"{r['src']} -> {r['tgt']}"
    )

base = N_EXAMPLES // len(top_pairs)
extra = N_EXAMPLES % len(top_pairs)
alloc = [base + (1 if i < extra else 0) for i in range(len(top_pairs))]

examples = []
for pair_info, n_take in zip(top_pairs, alloc):
    if n_take <= 0:
        continue
    picked = _top_trials_for_pair(pair_info, n_take=n_take)
    if len(picked) < n_take:
        print(
            f"  {pair_info['subject']}: only {len(picked)} positive-RPE trials "
            f"(wanted {n_take})"
        )
    examples.extend(picked)

examples = examples[:N_EXAMPLES]
if len(examples) == 0:
    raise RuntimeError("No manuscript example trials selected")

print(f"Selected {len(examples)} example trials across {len(top_pairs)} subjects")

hpc_color = "#7b68ee"
acc_color = "#222222"
raw_color = "#b0b0b0"
xlab = "Time from cue (ms)" if EVENT == "cue_on" else "Time from feedback (ms)"

n_cols = 5
n_pages = int(np.ceil(len(examples) / n_cols))
stem = f"ppc_hpc_acc_{EVENT}_manuscript_examples"

for page in range(n_pages):
    i0 = page * n_cols
    i1 = min((page + 1) * n_cols, len(examples))
    page_examples = examples[i0:i1]
    page_cols = len(page_examples)

    fig, axes = plt.subplots(
        2,
        page_cols,
        figsize=(5.0 * page_cols, 4.8),
        sharex=True,
        squeeze=False,
    )

    for col, ex in enumerate(page_examples):
        t = ex["times"] * 1000.0
        hpc_raw = ex["hpc_raw"]
        acc_raw = ex["acc_raw"]
        hpc_f = ex["hpc_f"]
        acc_f = ex["acc_f"]

        ax_h = axes[0, col]
        ax_a = axes[1, col]

        ax_h.plot(t, hpc_raw, color=raw_color, lw=0.9, alpha=0.9)
        ax_h.plot(t, hpc_f, color=hpc_color, lw=1.6)
        ax_h.axhline(0.0, color="0.85", lw=0.8)
        ax_h.set_ylabel(f"{ex['src']}\n(uV)", color=hpc_color)

        ax_a.plot(t, acc_raw, color=raw_color, lw=0.9, alpha=0.9)
        ax_a.plot(t, acc_f, color=acc_color, lw=1.6)
        ax_a.axhline(0.0, color="0.85", lw=0.8)
        ax_a.set_ylabel(f"{ex['tgt']}\n(uV)", color=acc_color)
        ax_a.set_xlabel(xlab)

        ax_h.set_title(
            f"{ex['subject']} | epoch {ex['epoch_idx'] + 1} | "
            f"PPC-z={ex['ppc_z_pair']:.2f} | phase-lock={ex['phase_lock']:.2f}",
            fontsize=9,
        )

        for ax in (ax_h, ax_a):
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            ax.tick_params(axis="both", which="major", length=4, width=1.0, bottom=True, left=True)

        y0 = np.nanmin(acc_raw)
        x0 = t.min() + 40
        x1 = x0 + 250
        amp = np.nanpercentile(np.abs(acc_raw), 75)
        if not np.isfinite(amp) or amp <= 0:
            amp = max(np.nanstd(acc_raw), 1e-6)
        ax_a.plot([x0, x1], [y0, y0], color="k", lw=1.2)
        ax_a.plot([x0, x0], [y0, y0 + amp], color="k", lw=1.2)
        ax_a.text((x0 + x1) / 2, y0, "250 ms", ha="center", va="top", fontsize=8)
        ax_a.text(x0, y0 + amp / 2, f"{amp:.1f} uV", ha="right", va="center", fontsize=8, rotation=90)

    fig.suptitle(
        f"HPC→ACC PPC example traces ({FMIN:.0f}-{FMAX:.0f} Hz overlay, positive RPE) | "
        f"top {len(top_pairs)} subjects | page {page + 1}/{n_pages}",
        y=1.02,
    )
    fig.tight_layout()

    pdf_path = os.path.join(plot_savepath, f"{stem}_page{page + 1:02d}.pdf")
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight")
    print("saved", pdf_path)
    plt.show()


## 8. HPC→target-region PPC bar+strip plot (restored manuscript-style panel)

Compute pair-level PPC-z for `HPC` as source to multiple targets (`ACC`, `AMY`, `OFC`, `dlPFC`, `dmPFC`) and recreate the bar+strip summary panel with per-target mixed-effects significance.


In [ ]:
import seaborn as sns

ROI_TARGETS = ["ACC", "AMY", "OFC", "dlPFC", "dmPFC"]


def cache_path_target(subj_id, roi_target, event=EVENT):
    stem = f"{subj_id}_{ROI_SOURCE}_{roi_target}_{event}_ppc_evsurr.npz"
    return os.path.join(savepath, stem)


def process_subject_target(subj_id, roi_target):
    path = cache_path_target(subj_id=subj_id, roi_target=roi_target, event=EVENT)
    if os.path.exists(path) and not OVERWRITE_CACHE:
        z = np.load(path, allow_pickle=True)
        return {k: z[k] for k in z.files}

    epoch_data = load_subject_epochs(
        subj_id=subj_id,
        base_dir=base_dir,
        event=EVENT,
        roi_source=ROI_SOURCE,
        roi_target=roi_target,
        tmin=None,
        tmax=None,
        target_sfreq=TARGET_SFREQ,
        day=DAY,
        fmin=None,
        fmax=None,
    )
    if epoch_data is None:
        return None

    source_ch, target_ch = _pool_roi_channels(epoch_data)
    if len(source_ch) == 0 or len(target_ch) == 0:
        return None

    epochs = epoch_data["epochs"].copy().crop(tmin=TMIN, tmax=TMAX)
    epochs_roi = epochs.copy().pick(source_ch + target_ch, verbose=False)
    source_idx = np.array([epochs_roi.ch_names.index(ch) for ch in source_ch], dtype=int)
    target_idx = np.array([epochs_roi.ch_names.index(ch) for ch in target_ch], dtype=int)
    indices = seed_target_indices(source_idx, target_idx)
    pair_src = np.array([epochs_roi.ch_names[i] for i in indices[0]], dtype=str)
    pair_tgt = np.array([epochs_roi.ch_names[i] for i in indices[1]], dtype=str)
    pair_label = np.array([f"{s}->{t}" for s, t in zip(pair_src, pair_tgt)], dtype=str)

    info = epochs_roi.info
    sfreq = float(info["sfreq"])

    trial_map = build_rpe_trial_map(
        beh_df,
        subj_id,
        n_epochs=len(epochs_roi),
        day=DAY,
        event=EVENT,
    )
    trial_map = trial_map[trial_map["alignment_valid"]].copy()

    out = {
        "subj_id": np.array(subj_id),
        "roi_target": np.array(roi_target),
        "pair_src": pair_src,
        "pair_tgt": pair_tgt,
        "pair_label": pair_label,
        "n_source": np.array(len(source_ch)),
        "n_target": np.array(len(target_ch)),
        "n_pairs": np.array(len(pair_label)),
    }

    n_ok = 0
    for cat in RPE_CATS:
        idx = (
            trial_map.loc[trial_map["rpe_cat"] == cat, "epoch_idx"]
            .astype(int)
            .drop_duplicates()
            .to_numpy()
        )
        out[f"n_trials_{cat}"] = np.array(len(idx))
        if len(idx) < MIN_TRIALS:
            continue

        arr = epochs_roi[idx].get_data(copy=True)

        def _conn_fn(x):
            return _compute_ppc(arr=x, info=info, indices=indices, sfreq=sfreq)

        ppc_true = _conn_fn(arr)
        ppc_z, surr_mean, surr_std = zscore_vs_evoked_surrogate(
            conn_fn=_conn_fn,
            data=arr,
            info=info,
            n_surrogates=N_SURROGATES,
            n_jobs=N_JOBS,
            seed=SEED,
        )
        out[f"ppc_{cat}"] = np.asarray(ppc_true, dtype=float)
        out[f"ppc_z_{cat}"] = np.asarray(ppc_z, dtype=float)
        out[f"surr_mean_{cat}"] = np.asarray(surr_mean, dtype=float)
        out[f"surr_std_{cat}"] = np.asarray(surr_std, dtype=float)
        n_ok += 1

    if n_ok == 0:
        return None

    np.savez_compressed(path, **out)
    return out


multi_results = {}
for roi_target in ROI_TARGETS:
    recs = {}
    for subj_id in tqdm(subjects, desc=f"PPC {ROI_SOURCE}->{roi_target}"):
        rec = process_subject_target(subj_id=subj_id, roi_target=roi_target)
        if rec is not None:
            recs[subj_id] = rec
    multi_results[roi_target] = recs
    print(f"{roi_target}: {len(recs)} subjects")

rows = []
for roi_target, recs in multi_results.items():
    for subj_id, rec in recs.items():
        pair_src = np.asarray(rec["pair_src"]).astype(str)
        pair_tgt = np.asarray(rec["pair_tgt"]).astype(str)
        pair_label = np.asarray(rec["pair_label"]).astype(str)
        for cat in RPE_CATS:
            col = f"ppc_z_{cat}"
            if col not in rec:
                continue
            vals = np.asarray(rec[col], dtype=float)
            for pi, v in enumerate(vals):
                if not np.isfinite(v):
                    continue
                rows.append(
                    {
                        "participant": str(subj_id),
                        "roi_source": ROI_SOURCE,
                        "roi_target": roi_target,
                        "event": EVENT,
                        "rpe_cat": cat,
                        "pair_label": str(pair_label[pi]),
                        "pair_uid": f"{subj_id}:{pair_label[pi]}",
                        "src_ch": str(pair_src[pi]),
                        "tgt_ch": str(pair_tgt[pi]),
                        "ppc_z": float(v),
                    }
                )

all_ppc_df = pd.DataFrame(rows)
all_ppc_csv = os.path.join(savepath, f"all_ppc_pairs_{EVENT}_{ROI_SOURCE}_to_targets.csv")
all_ppc_df.to_csv(all_ppc_csv, index=False)
print("saved", all_ppc_csv, "rows:", len(all_ppc_df))

plot_df = all_ppc_df[
    all_ppc_df["event"].eq(EVENT)
    & all_ppc_df["roi_source"].eq(ROI_SOURCE)
    & all_ppc_df["rpe_cat"].isin(["negative", "neutral", "positive"])
].copy()

if len(plot_df) == 0:
    print("No rows for target-region PPC plot")
else:
    cats = ["negative", "neutral", "positive"]
    colors = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}
    target_order = [t for t in ROI_TARGETS if t in set(plot_df["roi_target"].dropna().astype(str))]

    stats_rows = []
    sig_by_target = {}
    for roi_target in target_order:
        dfr = plot_df[plot_df["roi_target"].eq(roi_target)].copy()
        if dfr["participant"].nunique() < 2:
            continue
        try:
            m = smf.mixedlm(
                "ppc_z ~ C(rpe_cat, Treatment(reference='positive'))",
                data=dfr,
                groups=dfr["participant"],
                vc_formula={"pair_uid": "0 + C(pair_uid)"},
                re_formula="1",
            ).fit(reml=True, method="lbfgs", maxiter=200)
            term = "C(rpe_cat, Treatment(reference='positive'))[T.negative]"
            pval = float(m.pvalues.get(term, np.nan))
            beta = float(m.params.get(term, np.nan))
            sig_by_target[roi_target] = pval
            stats_rows.append(
                {
                    "roi_target": roi_target,
                    "n_rows": len(dfr),
                    "n_subjects": dfr["participant"].nunique(),
                    "beta_negative_vs_positive": beta,
                    "p_negative_vs_positive": pval,
                }
            )
        except Exception as exc:
            stats_rows.append(
                {
                    "roi_target": roi_target,
                    "n_rows": len(dfr),
                    "n_subjects": dfr["participant"].nunique(),
                    "beta_negative_vs_positive": np.nan,
                    "p_negative_vs_positive": np.nan,
                    "error": str(exc),
                }
            )

    stats_df = pd.DataFrame(stats_rows)
    stats_csv = os.path.join(savepath, f"all_ppc_targets_mixedlm_{EVENT}.csv")
    stats_df.to_csv(stats_csv, index=False)
    print("saved", stats_csv)
    display(stats_df)

    for roi_target in target_order:
        dfr = plot_df[plot_df["roi_target"].eq(roi_target)].copy()
        if len(dfr) == 0:
            continue

        target_means = (
            dfr.groupby(["rpe_cat"], observed=False)["ppc_z"]
            .agg(["mean", "sem"])
            .reset_index()
        )
        target_subj_means = (
            dfr.groupby(["participant", "rpe_cat"], observed=False)["ppc_z"]
            .mean()
            .reset_index()
        )

        fig, ax = plt.subplots(1, 1, figsize=(5.6, 5.0))
        x = np.arange(len(cats), dtype=float)
        bar_width = 0.55

        for i, cat in enumerate(cats):
            sub = target_means[target_means["rpe_cat"] == cat]
            if len(sub) == 0:
                continue
            sem = sub["sem"].iloc[0]
            ax.bar(
                x[i],
                sub["mean"].iloc[0],
                bar_width,
                yerr=sem if np.isfinite(sem) else 0.0,
                color=colors[cat],
                alpha=0.88,
                capsize=2.5,
                edgecolor="none",
                zorder=2,
            )

        sns.stripplot(
            data=target_subj_means,
            x="rpe_cat",
            y="ppc_z",
            order=cats,
            palette=colors,
            alpha=0.62,
            edgecolor="black",
            linewidth=0.3,
            size=5,
            ax=ax,
            zorder=10,
            jitter=0.12,
        )

        ax.axhline(0, color="black", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(cats)
        ax.tick_params(axis="both", which="major", length=4, width=1.0, bottom=True, left=True)
        ax.set_ylabel("Theta PPC z")
        ax.set_xlabel("")
        ax.text(0.44, -0.14, "HPC", transform=ax.transAxes, color="#7b68ee", fontsize=18, ha="right", va="center")
        ax.text(0.46, -0.14, f"\u2192 {roi_target}", transform=ax.transAxes, color="black", fontsize=15, ha="left", va="center")
        if EVENT == "cue_on":
            ax.text(0.985, 0.02, "recognition cue", transform=ax.transAxes, ha="right", va="bottom", fontsize=10)

        p = sig_by_target.get(roi_target, np.nan)
        if np.isfinite(p) and p < 0.05:
            yvals = []
            for cat in cats:
                sub = target_means[target_means["rpe_cat"] == cat]
                if len(sub):
                    m = float(sub["mean"].iloc[0])
                    s = float(sub["sem"].iloc[0]) if np.isfinite(sub["sem"].iloc[0]) else 0.0
                    yvals.append(m + s)
            if len(yvals):
                y = max(yvals) + 0.08
                x0, x1 = x[0], x[2]
                ax.plot([x0, x0, x1, x1], [y - 0.015, y, y, y - 0.015], color="k", lw=1.1)
                ax.text((x0 + x1) / 2.0, y + 0.015, "*", ha="center", va="bottom", fontsize=15)

        n_subj = dfr["participant"].nunique()
        ax.set_title(f"{EVENT} | {ROI_SOURCE}\u2192{roi_target} | n={n_subj} subjects")
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

        plt.tight_layout()

        plot_pdf = os.path.join(plot_savepath, f"ppc_bar_strip_{EVENT}_{ROI_SOURCE}_to_{roi_target}.pdf")
        fig.savefig(plot_pdf, dpi=300, bbox_inches="tight", facecolor="white")
        print("saved", plot_pdf)
        plt.show()




## 9–10. Trial-wise PSI: hit/miss × RPE (`phase_slope_index_time`)

Trial-level HPC→ACC theta PSI (4–8 Hz) via [`phase_slope_index_time`](https://mne.tools/mne-connectivity/stable/generated/mne_connectivity.phase_slope_index_time.html) with per-epoch cut-and-reverse surrogate z-scoring. **One row per trial × channel pair** enters the mixed models (same rows used for plots).

- **Section 9:** positive-RPE cue trials, hit vs miss (`psi_z ~ hit_miss`)
- **Section 10:** full `rpe_cat × hit_miss` interaction

Positive PSI ⇒ HPC leads ACC. Requires `EVENT='cue_on'`. After code changes set `OVERWRITE_CACHE=True` once; runtime is much longer than batch PPC because surrogates are computed per trial.


In [ ]:
from membandit.granger import build_hit_miss_trial_map

HIT_MISS_LEVELS = ["miss", "hit"]


def _symmetric_pad(epochs, tmin, tmax, pad):
    left = float(tmin) - float(epochs.times[0])
    right = float(epochs.times[-1]) - float(tmax)
    pad_use = float(min(pad, max(0.0, left), max(0.0, right)))
    epochs.crop(tmin=float(tmin) - pad_use, tmax=float(tmax) + pad_use)
    return epochs, pad_use


def cache_path_trial_psi(subj_id):
    return os.path.join(savepath_psi, f"{subj_id}_HPC_ACC_{EVENT}_psi_trial_hitmiss.npz")


def process_subject_trial_psi(subj_id):
    path = cache_path_trial_psi(subj_id)
    if os.path.exists(path) and not OVERWRITE_CACHE:
        z = np.load(path, allow_pickle=True)
        return {k: z[k] for k in z.files}

    epoch_data = load_subject_epochs(
        subj_id=subj_id,
        base_dir=base_dir,
        event=EVENT,
        roi_source=ROI_SOURCE,
        roi_target=ROI_TARGET,
        tmin=None,
        tmax=None,
        target_sfreq=TARGET_SFREQ,
        day=DAY,
        fmin=None,
        fmax=None,
    )
    if epoch_data is None:
        return None

    hpc, acc = _pool_roi_channels(epoch_data)
    if len(hpc) == 0 or len(acc) == 0:
        print(f"  skip {subj_id}: missing HPC ({len(hpc)}) or ACC ({len(acc)})")
        return None

    epochs = epoch_data["epochs"].copy()
    epochs, pad_use = _symmetric_pad(epochs, TMIN, TMAX, PAD)
    epochs_roi = epochs.copy().pick(hpc + acc, verbose=False)

    hpc_idx = np.array([epochs_roi.ch_names.index(ch) for ch in hpc], dtype=int)
    acc_idx = np.array([epochs_roi.ch_names.index(ch) for ch in acc], dtype=int)
    indices = seed_target_indices(hpc_idx, acc_idx)
    pair_src = np.array([epochs_roi.ch_names[i] for i in indices[0]], dtype=str)
    pair_tgt = np.array([epochs_roi.ch_names[i] for i in indices[1]], dtype=str)
    pair_label = np.array([f"{s}->{t}" for s, t in zip(pair_src, pair_tgt)], dtype=str)
    sfreq = float(epochs_roi.info["sfreq"])

    rpe_map = build_rpe_trial_map(
        beh_df,
        subj_id,
        n_epochs=len(epochs_roi),
        day=DAY,
        event=EVENT,
    )
    hm_map = build_hit_miss_trial_map(
        beh_df,
        subj_id,
        n_epochs=len(epochs_roi),
        day=DAY,
        event=EVENT,
    )

    use_cols_hm = [
        c for c in ["epoch_idx", "alignment_valid", "hit_miss", "hits"] if c in hm_map.columns
    ]
    trial_map = rpe_map.merge(
        hm_map[use_cols_hm],
        on="epoch_idx",
        how="inner",
        suffixes=("_rpe", "_hm"),
    )

    align_rpe = (
        trial_map["alignment_valid_rpe"]
        if "alignment_valid_rpe" in trial_map.columns
        else trial_map.get("alignment_valid", True)
    )
    align_hm = (
        trial_map["alignment_valid_hm"]
        if "alignment_valid_hm" in trial_map.columns
        else trial_map.get("alignment_valid", True)
    )

    trial_map = trial_map[
        align_rpe.astype(bool)
        & align_hm.astype(bool)
        & trial_map["rpe_cat"].isin(RPE_CATS)
        & trial_map["hit_miss"].isin(HIT_MISS_LEVELS)
    ].copy()
    trial_map = trial_map.drop_duplicates(subset=["epoch_idx"], keep="first")

    if trial_map.empty:
        print(f"  skip {subj_id}: no aligned RPE × hit/miss trials")
        return None

    idx = trial_map["epoch_idx"].astype(int).to_numpy()
    epochs_lab = epochs_roi[idx]

    psi_raw = compute_epoch_psi(
        epochs_lab,
        indices,
        sfreq,
        freqs=CWT_FREQS,
        fmin=FMIN,
        fmax=FMAX,
        n_cycles=CWT_N_CYCLES,
        padding=pad_use,
        n_jobs=N_JOBS,
    )
    psi_z, surr_mean, surr_std = zscore_epoch_psi(
        psi_raw,
        epochs_lab,
        indices,
        sfreq,
        freqs=CWT_FREQS,
        fmin=FMIN,
        fmax=FMAX,
        n_cycles=CWT_N_CYCLES,
        padding=pad_use,
        n_surrogates=N_SURROGATES,
        seed=SEED,
        n_jobs=N_JOBS,
    )

    out = {
        "subj_id": np.array(subj_id),
        "epoch_idx": idx.astype(int),
        "rpe_cat": trial_map["rpe_cat"].astype(str).to_numpy(),
        "hit_miss": trial_map["hit_miss"].astype(str).to_numpy(),
        "pair_src": pair_src,
        "pair_tgt": pair_tgt,
        "pair_label": pair_label,
        "n_pairs": np.array(len(pair_label)),
        "n_hpc": np.array(len(hpc)),
        "n_acc": np.array(len(acc)),
        "n_trials": np.array(len(idx)),
        "pad_use": np.array(pad_use),
        "psi_raw": np.asarray(psi_raw, dtype=float),
        "psi_z": np.asarray(psi_z, dtype=float),
        "surr_mean": np.asarray(surr_mean, dtype=float),
        "surr_std": np.asarray(surr_std, dtype=float),
    }
    np.savez_compressed(path, **out)
    return out


def trial_records_to_df(results):
    rows = []
    for subj_id, rec in results.items():
        cats = np.asarray(rec["rpe_cat"]).astype(str)
        hm_vals = np.asarray(rec["hit_miss"]).astype(str)
        epoch_idx = np.asarray(rec["epoch_idx"], dtype=int)
        psi_z = np.asarray(rec["psi_z"], dtype=float)
        psi_raw = np.asarray(rec["psi_raw"], dtype=float)
        pair_src = np.asarray(rec["pair_src"]).astype(str)
        pair_tgt = np.asarray(rec["pair_tgt"]).astype(str)
        pair_label = np.asarray(rec["pair_label"]).astype(str)

        if psi_z.ndim != 2:
            raise RuntimeError(f"{subj_id}: expected pair-level psi_z with shape (n_trials, n_pairs)")

        for ei in range(len(epoch_idx)):
            for pi in range(psi_z.shape[1]):
                if not np.isfinite(psi_z[ei, pi]):
                    continue
                plab = str(pair_label[pi])
                rows.append(
                    {
                        "subject": str(subj_id),
                        "epoch_idx": int(epoch_idx[ei]),
                        "pair_label": plab,
                        "pair_uid": f"{subj_id}:{plab}",
                        "src_ch": str(pair_src[pi]),
                        "tgt_ch": str(pair_tgt[pi]),
                        "rpe_cat": str(cats[ei]),
                        "hit_miss": str(hm_vals[ei]),
                        "psi_z": float(psi_z[ei, pi]),
                        "psi_raw": float(psi_raw[ei, pi]),
                    }
                )
    return pd.DataFrame(rows)


def _subj_cell_table(df, z_col):
    return (
        df.groupby(["subject", "rpe_cat", "hit_miss"], observed=False)[z_col]
        .agg(
            z_mean="mean",
            z_sem=lambda s: (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 else np.nan,
            n_obs="count",
        )
        .reset_index()
    )


if EVENT != "cue_on":
    print("Sections 9–10 require EVENT='cue_on'. Set EVENT and rerun.")
else:
    trial_psi_results = {}
    subjects_psi = list(subjects[:1] if SMOKE else subjects)
    for subj_id in tqdm(subjects_psi, desc="Trial-wise PSI (RPE × hit/miss)"):
        rec = process_subject_trial_psi(subj_id)
        if rec is not None:
            trial_psi_results[subj_id] = rec

    print("subjects with trial-wise PSI cache:", len(trial_psi_results))

    df_trial_psi = trial_records_to_df(trial_psi_results)
    if len(df_trial_psi):
        print(
            "trial×pair rows:",
            len(df_trial_psi),
            "subjects:",
            df_trial_psi["subject"].nunique(),
            "pair_uid:",
            df_trial_psi["pair_uid"].nunique(),
        )
        print("cells (trial×pair counts):")
        print(df_trial_psi.groupby(["rpe_cat", "hit_miss"], observed=False).size().to_string())
        print("subjects per cell:")
        print(
            df_trial_psi.groupby(["rpe_cat", "hit_miss"], observed=False)["subject"]
            .nunique()
            .unstack(fill_value=0)
            .to_string()
        )
    else:
        print("No trial-wise PSI rows available")


In [ ]:
# ## 9. Positive-RPE cue trials: hit vs miss PSI

if EVENT != "cue_on" or "df_trial_psi" not in globals() or len(df_trial_psi) == 0:
    print("Need df_trial_psi from the previous cell (EVENT='cue_on').")
else:
    df_hm_psi = df_trial_psi.loc[df_trial_psi["rpe_cat"].eq("positive")].copy()
    hm_csv = os.path.join(savepath_psi, f"psi_trials_positiveRPE_hitmiss_{EVENT}.csv")
    df_hm_psi.to_csv(hm_csv, index=False)
    print("saved", hm_csv, "rows:", len(df_hm_psi))

    if len(df_hm_psi) == 0:
        print("No positive-RPE trial rows for mixed model")
    else:
        print(
            "subjects:",
            df_hm_psi["subject"].nunique(),
            "pair_uid:",
            df_hm_psi["pair_uid"].nunique(),
        )

        m_hm = smf.mixedlm(
            "psi_z ~ C(hit_miss, Treatment('miss'))",
            data=df_hm_psi,
            groups=df_hm_psi["subject"],
            vc_formula={"pair_uid": "0 + C(pair_uid)"},
            re_formula="1",
        ).fit(reml=True, method="lbfgs", maxiter=500)
        print(m_hm.summary())

        term = "C(hit_miss, Treatment('miss'))[T.hit]"
        print(
            f"Hit vs miss effect: beta={m_hm.params.get(term, np.nan):.4f}, "
            f"p={m_hm.pvalues.get(term, np.nan):.5f}"
        )

        subj_hm_wide = (
            df_hm_psi.groupby(["subject", "hit_miss"], observed=False)["psi_z"]
            .mean()
            .unstack("hit_miss")
            .reindex(columns=HIT_MISS_LEVELS)
        )

        x = np.arange(2, dtype=float)
        fig, ax = plt.subplots(figsize=(4.2, 4.6))
        hm_colors = {"miss": "#d62728", "hit": "#2ca02c"}

        for hm_i, hm in enumerate(HIT_MISS_LEVELS):
            block = (
                df_hm_psi.loc[df_hm_psi["hit_miss"].eq(hm)]
                .groupby("subject", observed=False)["psi_z"]
                .agg(
                    z_mean="mean",
                    z_sem=lambda s: (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 else np.nan,
                    n_obs="count",
                )
                .reset_index()
            )
            jitter = np.linspace(-0.04, 0.04, len(block))
            for j, row in block.iterrows():
                if not np.isfinite(row["z_mean"]):
                    continue
                xdot = float(x[hm_i] + jitter[j])
                ax.scatter(xdot, float(row["z_mean"]), color="0.35", s=20, alpha=0.65, zorder=5)
                if int(row["n_obs"]) > 1 and np.isfinite(row["z_sem"]):
                    ax.errorbar(
                        xdot,
                        float(row["z_mean"]),
                        yerr=float(row["z_sem"]),
                        fmt="none",
                        ecolor="0.35",
                        elinewidth=0.9,
                        capsize=0,
                        alpha=0.65,
                        zorder=4,
                    )

        for subj in subj_hm_wide.index:
            ys = subj_hm_wide.loc[subj, HIT_MISS_LEVELS].to_numpy(dtype=float)
            if np.isfinite(ys).all():
                ax.plot(x, ys, color="0.78", lw=0.9, alpha=0.9, zorder=3)

        grand = subj_hm_wide.mean(axis=0, skipna=True)
        n_subj = subj_hm_wide.notna().sum(axis=0).clip(lower=1)
        sem = subj_hm_wide.std(axis=0, ddof=1, skipna=True) / np.sqrt(n_subj)
        for hm_i, hm in enumerate(HIT_MISS_LEVELS):
            if hm not in grand.index or not np.isfinite(grand[hm]):
                continue
            ax.errorbar(
                x[hm_i],
                float(grand[hm]),
                yerr=float(sem[hm]) if np.isfinite(sem[hm]) else 0.0,
                fmt="o",
                ms=9,
                capsize=0,
                color=hm_colors[hm],
                label=hm,
                zorder=6,
            )

        ax.axhline(0.0, color="k", ls="--", lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(HIT_MISS_LEVELS)
        ax.set_ylabel("Theta PSI z")
        ax.set_title(f"Positive-RPE cue trials · hit vs miss · N={subj_hm_wide.shape[0]}")
        if EVENT == "cue_on":
            ax.text(0.98, 0.02, "recognition cue", transform=ax.transAxes, ha="right", va="bottom", fontsize=9)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        fig.tight_layout()

        hm_plot_pdf = os.path.join(plot_savepath, f"psi_positiveRPE_hitmiss_{EVENT}.pdf")
        fig.savefig(hm_plot_pdf, dpi=300, bbox_inches="tight")
        print("saved", hm_plot_pdf)
        plt.show()

        complete = subj_hm_wide.dropna(subset=HIT_MISS_LEVELS)
        if len(complete) >= 2:
            stat = wilcoxon(complete["hit"], complete["miss"], zero_method="wilcox", alternative="two-sided")
            print(f"Paired Wilcoxon (complete-case) hit-vs-miss: W={stat.statistic:.3f}, p={stat.pvalue:.5f}")

        print(
            "\n# R / lme4 starter code (positive-RPE cue hit-vs-miss PSI)\n"
            "library(lme4)\n\n"
            f'df <- read.csv("{hm_csv}")\n'
            'df$hit_miss <- relevel(factor(df$hit_miss), ref = "miss")\n\n'
            "m <- lmer(psi_z ~ hit_miss + (1 | subject) + (1 | pair_uid), data = df)\n"
            "summary(m)\n"
        )


In [ ]:
# ## 10. RPE × hit/miss interaction model (trial-wise PSI)

if EVENT != "cue_on" or "df_trial_psi" not in globals() or len(df_trial_psi) == 0:
    print("Need df_trial_psi from Section 9–10 pipeline cell.")
else:
    csv_rpe_hm = os.path.join(savepath_psi, f"psi_trials_rpe_x_hitmiss_{EVENT}.csv")
    df_trial_psi.to_csv(csv_rpe_hm, index=False)
    print("saved", csv_rpe_hm, "rows:", len(df_trial_psi))

    print(
        "subjects:",
        df_trial_psi["subject"].nunique(),
        "pair_uid:",
        df_trial_psi["pair_uid"].nunique(),
        "cells:",
        df_trial_psi.groupby(["rpe_cat", "hit_miss"], observed=False).size().to_dict(),
    )

    formula = "psi_z ~ C(rpe_cat, Treatment('positive')) * C(hit_miss, Treatment('miss'))"
    re_formula = "1 + C(rpe_cat, Treatment('positive'))"

    m_rpe_hm = None
    fit_note = ""
    try:
        m_try = smf.mixedlm(
            formula,
            data=df_trial_psi,
            groups=df_trial_psi["subject"],
            re_formula=re_formula,
            vc_formula={"pair_uid": "0 + C(pair_uid)"},
        ).fit(reml=True, method="lbfgs", maxiter=500)
        if not bool(getattr(m_try, "converged", False)):
            raise RuntimeError("random-slope model did not converge")
        m_rpe_hm = m_try
        fit_note = "Using random-slope model across subjects for rpe_cat"
    except Exception as err:
        print("Random-slope fit failed; falling back to random-intercept subject model")
        print("reason:", repr(err))
        m_rpe_hm = smf.mixedlm(
            formula,
            data=df_trial_psi,
            groups=df_trial_psi["subject"],
            re_formula="1",
            vc_formula={"pair_uid": "0 + C(pair_uid)"},
        ).fit(reml=True, method="lbfgs", maxiter=500)
        fit_note = "Fallback model: subject random intercept only"

    print(fit_note)
    print(m_rpe_hm.summary())

    terms_focus = [
        "C(rpe_cat, Treatment('positive'))[T.negative]",
        "C(rpe_cat, Treatment('positive'))[T.neutral]",
        "C(hit_miss, Treatment('miss'))[T.hit]",
        "C(rpe_cat, Treatment('positive'))[T.negative]:C(hit_miss, Treatment('miss'))[T.hit]",
        "C(rpe_cat, Treatment('positive'))[T.neutral]:C(hit_miss, Treatment('miss'))[T.hit]",
    ]
    print("\nFocused fixed-effects readout:")
    for t in terms_focus:
        b = m_rpe_hm.params.get(t, np.nan)
        p = m_rpe_hm.pvalues.get(t, np.nan)
        print(f"  {t}: beta={b:.4f}, p={p:.5f}")

    subj_cell = _subj_cell_table(df_trial_psi, "psi_z")

    x = np.arange(len(RPE_CATS), dtype=float)
    width = 0.34
    offset = {"miss": -width / 2, "hit": width / 2}
    hm_colors = {"miss": "#d62728", "hit": "#2ca02c"}

    fig, ax = plt.subplots(figsize=(7.6, 4.8))
    for hm in HIT_MISS_LEVELS:
        means = []
        sems = []
        for cat in RPE_CATS:
            block = subj_cell.loc[
                subj_cell["rpe_cat"].eq(cat) & subj_cell["hit_miss"].eq(hm)
            ]
            vals = block["z_mean"].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            means.append(np.nanmean(vals) if len(vals) else np.nan)
            sems.append((np.nanstd(vals, ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else np.nan)

        xpos = x + offset[hm]
        ax.bar(
            xpos,
            means,
            width=width,
            color=hm_colors[hm],
            alpha=0.72,
            edgecolor="none",
            label=hm,
            zorder=2,
        )
        ax.errorbar(
            xpos,
            means,
            yerr=[0.0 if not np.isfinite(v) else float(v) for v in sems],
            fmt="none",
            ecolor="k",
            elinewidth=1.0,
            capsize=0,
            zorder=4,
        )

        for i, cat in enumerate(RPE_CATS):
            block = subj_cell.loc[
                subj_cell["rpe_cat"].eq(cat) & subj_cell["hit_miss"].eq(hm)
            ]
            if block.empty:
                continue
            jitter = np.linspace(-0.05, 0.05, len(block))
            for j, (_, row) in enumerate(block.iterrows()):
                if not np.isfinite(row["z_mean"]):
                    continue
                xdot = float(xpos[i] + jitter[j])
                ydot = float(row["z_mean"])
                ax.scatter(
                    xdot,
                    ydot,
                    s=20,
                    color="0.35",
                    alpha=0.65,
                    linewidths=0.4,
                    edgecolors="white",
                    zorder=5,
                )
                if int(row["n_obs"]) > 1 and np.isfinite(row["z_sem"]):
                    ax.errorbar(
                        xdot,
                        ydot,
                        yerr=float(row["z_sem"]),
                        fmt="none",
                        ecolor="0.35",
                        elinewidth=0.9,
                        capsize=0,
                        alpha=0.65,
                        zorder=4,
                    )

    ax.axhline(0.0, color="k", ls="--", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(list(RPE_CATS))
    ax.set_ylabel("Theta PSI z")
    ax.set_xlabel("RPE category")
    ax.set_title(f"{EVENT} | HPC→ACC | RPE × hit/miss (trial-wise PSI)")
    ax.legend(frameon=False, title="hit_miss")
    if EVENT == "cue_on":
        ax.text(0.98, 0.02, "recognition cue", transform=ax.transAxes, ha="right", va="bottom", fontsize=9)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()

    rpe_hm_plot_pdf = os.path.join(plot_savepath, f"psi_rpe_x_hitmiss_{EVENT}.pdf")
    fig.savefig(rpe_hm_plot_pdf, dpi=300, bbox_inches="tight")
    print("saved", rpe_hm_plot_pdf)
    plt.show()

    print(
        "\n# R / lme4 starter code (RPE x hit/miss cue PSI)\n"
        "library(lme4)\n\n"
        f'df <- read.csv("{csv_rpe_hm}")\n'
        'df$rpe_cat <- relevel(factor(df$rpe_cat), ref = "positive")\n'
        'df$hit_miss <- relevel(factor(df$hit_miss), ref = "miss")\n\n'
        "m <- lmer(psi_z ~ rpe_cat * hit_miss + (1 + rpe_cat | subject) + (1 | pair_uid), data = df)\n"
        "summary(m)\n"
    )


## 11. HPC→ACC PSI bar+strip by RPE category

Uses the **Section 9–10 trial-wise PSI cache** (`*_psi_trial_hitmiss.npz`) — no recomputation. Trial×pair `psi_z` values are averaged within each `(subject, rpe_cat, pair)` cell, then summarized with the same bar+strip layout as Section 8 PPC.

In [ ]:
import seaborn as sns


def load_trial_psi_for_rpe_plot():
    """Pair×RPE table from Section 9–10 trial PSI cache (no recomputation)."""
    if "df_trial_psi" in globals() and len(df_trial_psi) > 0:
        src = df_trial_psi.copy()
    elif "trial_records_to_df" in globals() and "cache_path_trial_psi" in globals():
        results = {}
        subjects_load = list(subjects[:1] if SMOKE else subjects)
        for subj_id in subjects_load:
            path = cache_path_trial_psi(subj_id)
            if os.path.exists(path):
                z = np.load(path, allow_pickle=True)
                results[subj_id] = {k: z[k] for k in z.files}
        src = trial_records_to_df(results)
    else:
        return pd.DataFrame()

    cats_keep = ["negative", "neutral", "positive"]
    src = src[src["rpe_cat"].isin(cats_keep)].copy()
    if len(src) == 0:
        return pd.DataFrame()

    pair_rpe = (
        src.groupby(
            ["subject", "rpe_cat", "pair_uid", "pair_label", "src_ch", "tgt_ch"],
            observed=False,
        )["psi_z"]
        .mean()
        .reset_index()
    )
    pair_rpe = pair_rpe.rename(columns={"subject": "participant"})
    pair_rpe["roi_source"] = ROI_SOURCE
    pair_rpe["roi_target"] = ROI_TARGET
    pair_rpe["event"] = EVENT
    return pair_rpe


if EVENT != "cue_on":
    print("Section 11 requires EVENT='cue_on'.")
else:
    plot_psi_df = load_trial_psi_for_rpe_plot()
    all_psi_csv = os.path.join(
        savepath_psi, f"all_psi_pairs_{EVENT}_{ROI_SOURCE}_to_{ROI_TARGET}_from_trial_cache.csv"
    )
    plot_psi_df.to_csv(all_psi_csv, index=False)
    print("saved", all_psi_csv, "rows:", len(plot_psi_df))

    if len(plot_psi_df) == 0:
        print("No rows for PSI RPE plot — run Section 9–10 trial PSI cell first.")
    else:
        cats = ["negative", "neutral", "positive"]
        colors = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}
        roi_target = ROI_TARGET

        psi_stats_rows = []
        psi_sig_by_target = {}
        dfr = plot_psi_df.copy()
        if dfr["participant"].nunique() >= 2:
            try:
                m = smf.mixedlm(
                    "psi_z ~ C(rpe_cat, Treatment(reference='positive'))",
                    data=dfr,
                    groups=dfr["participant"],
                    vc_formula={"pair_uid": "0 + C(pair_uid)"},
                    re_formula="1",
                ).fit(reml=True, method="lbfgs", maxiter=200)
                term = "C(rpe_cat, Treatment(reference='positive'))[T.negative]"
                pval = float(m.pvalues.get(term, np.nan))
                beta = float(m.params.get(term, np.nan))
                psi_sig_by_target[roi_target] = pval
                psi_stats_rows.append(
                    {
                        "roi_target": roi_target,
                        "n_rows": len(dfr),
                        "n_subjects": dfr["participant"].nunique(),
                        "beta_negative_vs_positive": beta,
                        "p_negative_vs_positive": pval,
                    }
                )
            except Exception as exc:
                psi_stats_rows.append(
                    {
                        "roi_target": roi_target,
                        "n_rows": len(dfr),
                        "n_subjects": dfr["participant"].nunique(),
                        "beta_negative_vs_positive": np.nan,
                        "p_negative_vs_positive": np.nan,
                        "error": str(exc),
                    }
                )

        psi_stats_df = pd.DataFrame(psi_stats_rows)
        psi_stats_csv = os.path.join(savepath_psi, f"psi_rpe_mixedlm_{EVENT}_{ROI_SOURCE}_to_{ROI_TARGET}.csv")
        psi_stats_df.to_csv(psi_stats_csv, index=False)
        print("saved", psi_stats_csv)
        display(psi_stats_df)

        r_stub = f'''
# R / lme4 starter code (HPC→ACC cue PSI × RPE)
library(lme4)

df <- read.csv("{all_psi_csv}")
df$rpe_cat <- relevel(factor(df$rpe_cat), ref = "positive")

# Preferred model: random slope of rpe_cat across subjects + pair random intercept
m <- lmer(psi_z ~ rpe_cat + (1 + rpe_cat | participant) + (1 | pair_uid), data = df)

# Closest match to Python MixedLM / fallback if singular:
# m <- lmer(psi_z ~ rpe_cat + (1 | participant) + (1 | pair_uid), data = df)

summary(m)
'''
        print(r_stub)

        target_means = (
            dfr.groupby(["rpe_cat"], observed=False)["psi_z"]
            .agg(["mean", "sem"])
            .reset_index()
        )
        target_subj_means = (
            dfr.groupby(["participant", "rpe_cat"], observed=False)["psi_z"]
            .mean()
            .reset_index()
        )

        fig, ax = plt.subplots(1, 1, figsize=(5.6, 5.0))
        x = np.arange(len(cats), dtype=float)
        bar_width = 0.55

        for i, cat in enumerate(cats):
            sub = target_means[target_means["rpe_cat"] == cat]
            if len(sub) == 0:
                continue
            sem = sub["sem"].iloc[0]
            ax.bar(
                x[i],
                sub["mean"].iloc[0],
                bar_width,
                yerr=sem if np.isfinite(sem) else 0.0,
                color=colors[cat],
                alpha=0.88,
                capsize=2.5,
                edgecolor="none",
                zorder=2,
            )

        sns.stripplot(
            data=target_subj_means,
            x="rpe_cat",
            y="psi_z",
            order=cats,
            palette=colors,
            alpha=0.62,
            edgecolor="black",
            linewidth=0.3,
            size=5,
            ax=ax,
            zorder=10,
            jitter=0.12,
        )

        ax.axhline(0, color="black", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(cats)
        ax.tick_params(axis="both", which="major", length=4, width=1.0, bottom=True, left=True)
        ax.set_ylabel("Theta PSI z")
        ax.set_xlabel("")
        ax.text(0.44, -0.14, "HPC", transform=ax.transAxes, color="#7b68ee", fontsize=18, ha="right", va="center")
        ax.text(0.46, -0.14, f"\u2192 {roi_target}", transform=ax.transAxes, color="black", fontsize=15, ha="left", va="center")
        if EVENT == "cue_on":
            ax.text(0.985, 0.02, "recognition cue", transform=ax.transAxes, ha="right", va="bottom", fontsize=10)

        p = psi_sig_by_target.get(roi_target, np.nan)
        if np.isfinite(p) and p < 0.05:
            yvals = []
            for cat in cats:
                sub = target_means[target_means["rpe_cat"] == cat]
                if len(sub):
                    mval = float(sub["mean"].iloc[0])
                    s = float(sub["sem"].iloc[0]) if np.isfinite(sub["sem"].iloc[0]) else 0.0
                    yvals.append(mval + s)
            if len(yvals):
                y = max(yvals) + 0.08
                x0, x1 = x[0], x[2]
                ax.plot([x0, x0, x1, x1], [y - 0.015, y, y, y - 0.015], color="k", lw=1.1)
                ax.text((x0 + x1) / 2.0, y + 0.015, "*", ha="center", va="bottom", fontsize=15)

        n_subj = dfr["participant"].nunique()
        ax.set_title(f"{EVENT} | {ROI_SOURCE}\u2192{roi_target} | n={n_subj} subjects")
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

        plt.tight_layout()

        plot_pdf = os.path.join(plot_savepath, f"psi_bar_strip_{EVENT}_{ROI_SOURCE}_to_{roi_target}.pdf")
        fig.savefig(plot_pdf, dpi=300, bbox_inches="tight", facecolor="white")
        print("saved", plot_pdf)
        plt.show()


In [ ]:
from scipy.stats import chi2, norm


def _fit_ml_interaction_models(df, z_col="psi_z"):
    formula_add = f"{z_col} ~ C(rpe_cat, Treatment('positive')) + C(hit_miss, Treatment('miss'))"
    formula_int = f"{z_col} ~ C(rpe_cat, Treatment('positive')) * C(hit_miss, Treatment('miss'))"
    vc = {"pair_uid": "0 + C(pair_uid)"}
    kwargs = dict(data=df, groups=df["subject"], vc_formula=vc)

    def _try(formula, re_formula):
        m = smf.mixedlm(formula, re_formula=re_formula, **kwargs).fit(
            reml=False, method="lbfgs", maxiter=500
        )
        if not bool(getattr(m, "converged", False)):
            raise RuntimeError("did not converge")
        return m

    try:
        m_add = _try(formula_add, "1 + C(rpe_cat, Treatment('positive'))")
        m_int = _try(formula_int, "1 + C(rpe_cat, Treatment('positive'))")
        spec = "subject random slope for rpe_cat"
    except Exception:
        m_add = _try(formula_add, "1")
        m_int = _try(formula_int, "1")
        spec = "subject random intercept only (slope model did not converge)"
    return m_add, m_int, spec


def _hit_minus_miss_simple_effects(m):
    fe = m.fe_params.copy()
    cov = m.cov_params().loc[fe.index, fe.index].to_numpy(dtype=float)
    hit = "C(hit_miss, Treatment('miss'))[T.hit]"
    neg = "C(rpe_cat, Treatment('positive'))[T.negative]:C(hit_miss, Treatment('miss'))[T.hit]"
    neu = "C(rpe_cat, Treatment('positive'))[T.neutral]:C(hit_miss, Treatment('miss'))[T.hit]"
    weights = {
        "positive": {hit: 1.0},
        "negative": {hit: 1.0, neg: 1.0},
        "neutral": {hit: 1.0, neu: 1.0},
    }
    rows = []
    for cat, wmap in weights.items():
        w = np.zeros(len(fe), dtype=float)
        for name, val in wmap.items():
            if name in fe.index:
                w[fe.index.get_loc(name)] = val
        beta = float(np.dot(w, fe.to_numpy(dtype=float)))
        se = float(np.sqrt(max(np.dot(w, cov.dot(w)), 0.0)))
        z = beta / se if se > 0 else np.nan
        p = 2.0 * norm.sf(abs(z)) if np.isfinite(z) else np.nan
        rows.append({"rpe_cat": cat, "beta": beta, "se": se, "p": p})
    return pd.DataFrame(rows)


def _sig(p):
    return "YES" if (np.isfinite(p) and p < 0.05) else "no"


if "df_trial_psi" not in globals() or len(df_trial_psi) == 0:
    print("Need df_trial_psi from the Section 9–10 pipeline cell.")
else:
    m_add, m_int, spec = _fit_ml_interaction_models(df_trial_psi, z_col="psi_z")
    lrt = float(2.0 * (m_int.llf - m_add.llf))
    lrt_df = int(m_int.k_fe - m_add.k_fe)
    lrt_p = float(chi2.sf(max(lrt, 0.0), max(lrt_df, 1)))
    simple = _hit_minus_miss_simple_effects(m_int)

    print("Does the hit-miss PSI difference depend on RPE category?")
    print(f"  interaction LRT: chi2({lrt_df}) = {lrt:.3f}, p = {lrt_p:.4f}   significant? {_sig(lrt_p)}")
    print(f"  random effects: {spec}")
    print()
    print("Within each RPE category, is hit PSI higher than miss PSI?")
    for _, r in simple.iterrows():
        print(
            f"  {r['rpe_cat']:>8s}:  hit-miss beta = {r['beta']:+.3f}   "
            f"p = {r['p']:.3f}   significant? {_sig(r['p'])}"
        )
